### Qualidade de Dados: Application

In [0]:
from pyspark.sql import functions as F

df = spark.table("credit_risk_pipeline.silver.application")
total_linhas = df.count()

# Completude: percentual de nulos por coluna
completude = df.select([
    (F.count(F.when(F.col(c).isNull(), c)) / total_linhas * 100).alias(c)
    for c in df.columns
])
display(completude)

# Unicidade: cada cliente deveria aparecer só uma vez
duplicados = df.groupBy("SK_ID_CURR").count().filter("count > 1")
print(f"Clientes duplicados: {duplicados.count()} de {total_linhas}")

# Acurácia e outliers: olhando a distribuição das colunas numéricas mais sensíveis
df.select("AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "idade_anos") \
  .summary("min", "25%", "50%", "75%", "max") \
  .show()

In [0]:
colunas_categoricas = [
    "NAME_CONTRACT_TYPE", "CODE_GENDER", "NAME_INCOME_TYPE",
    "NAME_EDUCATION_TYPE", "NAME_FAMILY_STATUS", "NAME_HOUSING_TYPE",
    "OCCUPATION_TYPE", "ORGANIZATION_TYPE"
]

for coluna in colunas_categoricas:
    qtd_xna = df.filter(F.col(coluna) == "XNA").count()
    if qtd_xna > 0:
        print(f"{coluna}: {qtd_xna} linhas com 'XNA'")

### Qualidade de Dados: Bureau, Previous Application e Installments

In [0]:
def checar_qualidade(nome_tabela, coluna_chave, colunas_numericas):
    df = spark.table(f"credit_risk_pipeline.silver.{nome_tabela}")
    total_linhas = df.count()

    print(f"--- {nome_tabela} ({total_linhas} linhas) ---")

    # Completude
    completude = df.select([
        (F.count(F.when(F.col(c).isNull(), c)) / total_linhas * 100).alias(c)
        for c in df.columns
    ])
    display(completude)

    # Unicidade (só quando a tabela tiver uma chave de uma linha por registro)
    if coluna_chave:
        duplicados = df.groupBy(coluna_chave).count().filter("count > 1")
        print(f"Duplicados em {coluna_chave}: {duplicados.count()} de {total_linhas}")

    # Outliers
    if colunas_numericas:
        df.select(colunas_numericas).summary("min", "25%", "50%", "75%", "max").show()


checar_qualidade("bureau", "SK_ID_BUREAU", ["DAYS_CREDIT", "CREDIT_DAY_OVERDUE", "AMT_CREDIT_SUM", "AMT_CREDIT_SUM_DEBT"])
checar_qualidade("previous_application", "SK_ID_PREV", ["AMT_APPLICATION", "AMT_CREDIT", "AMT_ANNUITY"])
checar_qualidade("installments_payments", None, ["AMT_INSTALMENT", "AMT_PAYMENT", "atraso_dias"])